In [1]:
import plotly.io as pio
import numpy as np
from fastembed import LateInteractionTextEmbedding
from fastembed import TextEmbedding
from qdrant_client import QdrantClient, models
import time

pio.renderers.default = "notebook"

2026-02-24 10:45:55.593347519 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


### Embedding Model

In [2]:
dense_model = TextEmbedding("BAAI/bge-small-en-v1.5")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
dense_model.embedding_size

384

In [4]:
dense_model.__dict__

{'model_name': 'BAAI/bge-small-en-v1.5',
 'cache_dir': None,
 'threads': None,
 '_local_files_only': False,
 '_embedding_size': 384,
 'model': <fastembed.text.onnx_embedding.OnnxTextEmbedding at 0x7f2322edf4d0>}

### Simple Embedding

In [5]:
document = """This study examines the environmental benefits of 
electric bus fleets in three major metropolitan areas over a 
two-year period. Our analysis shows that electric buses reduce 
carbon emissions by an average of 65% compared to traditional 
diesel buses, while also decreasing noise pollution in urban 
centers by 40 decibels."""

In [6]:
document_embeddings = next(dense_model.passage_embed([document]))

In [7]:
document_embeddings.shape

(384,)

In [8]:
query = "advantages of EV cars"

In [9]:
query_embeddings = next(dense_model.query_embed([query]))

In [10]:
query_embeddings.shape

(384,)

### Qdrant

In [12]:
collection_name = "qdrant-tests"
dense_vector_name = "BAAI-bge-small-en-v1.5"

In [13]:
client = QdrantClient("http://localhost:6333")

In [14]:
client.delete_collection(collection_name)

True

In [15]:
client.create_collection(
    collection_name,
    vectors_config={
        dense_vector_name: models.VectorParams(
            size=dense_model.embedding_size,
            distance=models.Distance.COSINE,
        ),       
    }
)

True

In [16]:
documents = [
    "Qdrant is a vector database designed for similarity search applications",
    "SQL databases use structured tables with predefined schemas for data storage",
    "Using Qdrant you can store embeddings and perform efficient searches",
    "Traditional SQL queries filter data using exact matches and joins",
    "Qdrant supports multi-vector configurations for late interaction models like ColBERT",
    "SQL performs well for transactional workloads but lacks semantic search capabilities",
    "The Qdrant client allows you to create collections with custom distance metrics",
    "Migrating from SQL to vector databases enables similarity-based retrieval at scale",
    "Qdrant's MaxSim comparator enables token-level similarity scoring for multi-vectors",
    "SQL databases struggle with high-dimensional embeddings unlike specialized vector stores",
]

In [17]:
client.upsert(
    collection_name,
    points=[
        models.PointStruct(
            id=i,
            vector={
                dense_vector_name: next(
                    dense_model.passage_embed([document])
                ),
            },
            payload={"text": document},
        )
        for i, document in enumerate(documents, start=1)
    ],
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

### Query

In [18]:
def dense_query(q: str, limit: int = 5) -> list[dict]:
    start_time = time.monotonic()
    embedding = next(dense_model.query_embed(q))
    end_time = time.monotonic()
    
    print("Dense vector generation time:", end_time - start_time)

    start_time = time.monotonic()
    result = client.query_points(
        collection_name,
        query=embedding,
        using=dense_vector_name,
        limit=limit,
        with_payload=True,
    )
    end_time = time.monotonic()
    print("Query time:", end_time - start_time)

    return [point.payload for point in result.points]    

In [19]:
query = "search performance in Qdrant"

In [20]:
dense_hits = dense_query(query)

Dense vector generation time: 0.008671473000049446
Query time: 0.015337346000251273


In [21]:
dense_hits

[{'text': 'Using Qdrant you can store embeddings and perform efficient searches'},
 {'text': 'Qdrant is a vector database designed for similarity search applications'},
 {'text': 'Qdrant supports multi-vector configurations for late interaction models like ColBERT'},
 {'text': 'SQL performs well for transactional workloads but lacks semantic search capabilities'},
 {'text': 'Migrating from SQL to vector databases enables similarity-based retrieval at scale'}]